# SS7 content regex match report - REG-REQ

Same setup as `ss7_regex_content_match.ipynb`: scans every raw SS7 CDR
file, keeps `message_type in [2, 3]` (MT_request / MO — the two message
types that actually carry content, see `ingestion/ss7.py`'s module
docstring), and searches decoded content for a regex pattern - here
`REG-REQ` (literal substring, case-sensitive), instead of the earlier
bracketed-token shape.

**Column mapping** (raw SS7 CSV → task spec names):

| task spec   | real column       |
|-------------|--------------------|
| messageType | `message_type`     |
| dcs         | `dcs`               |
| callingGT   | `calling_gt`        |
| calledGT    | `called_gt`         |
| smsPayload  | `decoded_content`   |
| timestamp   | `time_stamp`        |

**Why `decoded_content` directly, no hex→ASCII step here**: `ingestion/ss7.py`
(`_decode_row`'s docstring) already verified `decoded_content` matches
`ingestion/dcs_codecs.py::decode_by_dcs()`'s own decode of the raw hex
`content`/`raw_user_data` field exactly, on a real multipart sample — unlike
SMPP, where `decoded_content` was found to leak UDH framing bytes and is
NOT trusted. So for SS7, reusing the upstream `decoded_content` column is
equivalent to decoding the hex payload ourselves. Coverage caveat: rows
where `decoded_content` is null but `content`/`raw_user_data` is not
(genuinely undecoded rows) are reported separately below, not silently
dropped.

Processes files one at a time (48 files, ~4GB raw) with only the needed
columns loaded, instead of concatenating the full corpus into memory.

In [4]:
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_SS7_DIR = PROJECT_ROOT / "data" / "raw" / "SS7"

PATTERN = re.compile(r"REG-REQ")
MESSAGE_TYPES_TO_CHECK = [2, 3]  # MT_request, MO - see markdown above

USECOLS = [
    "time_stamp", "message_type", "dcs", "calling_gt", "called_gt",
    "decoded_content", "content", "raw_user_data", "reference",
]

raw_files = sorted(RAW_SS7_DIR.glob("*/*.csv"))
print(f"{len(raw_files)} raw SS7 files found under {RAW_SS7_DIR}")

48 raw SS7 files found under c:\Users\IshitaGodani\Documents\projects\spam-detection-prototype\data\raw\SS7


In [5]:
match_chunks = []
total_rows_seen = 0
total_filtered_rows = 0  # message_type in [2, 3]
total_undecoded_filtered_rows = 0  # filtered rows with no decoded_content
# but real content/raw_user_data present - NOT searched, reported as a gap

for i, path in enumerate(raw_files, start=1):
    df = pd.read_csv(path, usecols=lambda c: c in USECOLS, low_memory=False)
    total_rows_seen += len(df)

    filtered = df[df["message_type"].isin(MESSAGE_TYPES_TO_CHECK)]
    total_filtered_rows += len(filtered)

    has_payload = filtered["content"].notna() | filtered["raw_user_data"].notna()
    undecoded = filtered["decoded_content"].isna() & has_payload
    total_undecoded_filtered_rows += int(undecoded.sum())

    hit = filtered["decoded_content"].str.contains(PATTERN, regex=True, na=False)
    if hit.any():
        match_chunks.append(filtered.loc[hit, [
            "time_stamp", "message_type", "dcs", "calling_gt", "called_gt",
            "decoded_content", "content", "reference",
        ]])

    if i % 10 == 0 or i == len(raw_files):
        print(f"[{i}/{len(raw_files)}] {path.name}: {len(df)} rows, "
              f"{len(filtered)} in scope, {len(match_chunks)} files with matches so far")

matches = (
    pd.concat(match_chunks, ignore_index=True)
    if match_chunks else
    pd.DataFrame(columns=["time_stamp", "message_type", "dcs", "calling_gt", "called_gt", "decoded_content", "content", "reference"])
)

print(f"\ntotal rows scanned: {total_rows_seen:,}")
print(f"rows in scope (message_type in {MESSAGE_TYPES_TO_CHECK}): {total_filtered_rows:,}")
print(f"in-scope rows with a real payload but no decoded_content (not searched): {total_undecoded_filtered_rows:,}")
print(f"total pattern matches: {len(matches):,}")

[10/48] stg_ss7_20260802_0900.csv: 266177 rows, 57851 in scope, 10 files with matches so far
[20/48] stg_ss7_20260802_1900.csv: 339103 rows, 76669 in scope, 20 files with matches so far
[30/48] stg_ss7_20260803_0500.csv: 157936 rows, 28456 in scope, 30 files with matches so far
[40/48] stg_ss7_20260803_1500.csv: 542639 rows, 157581 in scope, 40 files with matches so far
[48/48] stg_ss7_20260803_2300.csv: 261057 rows, 53615 in scope, 48 files with matches so far

total rows scanned: 14,685,436
rows in scope (message_type in [2, 3]): 3,417,425
in-scope rows with a real payload but no decoded_content (not searched): 40
total pattern matches: 7,153


## Summary tables

In [6]:
print("total match count:", len(matches))

total match count: 7153


In [7]:
print("match count by message_type:")
matches["message_type"].value_counts().sort_index()

match count by message_type:


message_type
3    7153
Name: count, dtype: int64

In [8]:
print("match count by dcs:")
matches["dcs"].value_counts().sort_index()

match count by dcs:


dcs
0    7153
Name: count, dtype: int64

In [9]:
calling_gt_counts = matches["calling_gt"].value_counts()
print(f"distinct callingGT: {calling_gt_counts.shape[0]}")
calling_gt_counts

distinct callingGT: 327


calling_gt
60120000060     323
6738400118      242
60120000061     211
6738400218      207
923330053007    206
               ... 
919892125680      1
8613254298        1
998909135999      1
905329372200      1
97333811012       1
Name: count, Length: 327, dtype: int64

In [10]:
called_gt_counts = matches["called_gt"].value_counts()
print(f"distinct calledGT: {called_gt_counts.shape[0]}")
called_gt_counts

distinct calledGT: 1


called_gt
60120000015    7153
Name: count, dtype: int64

In [11]:
gt_pair_counts = matches.groupby(["calling_gt", "called_gt"]).size().sort_values(ascending=False)
print(f"distinct (callingGT, calledGT) pairs: {gt_pair_counts.shape[0]}")
gt_pair_counts

distinct (callingGT, calledGT) pairs: 327


calling_gt    called_gt  
60120000060   60120000015    323
6738400118    60120000015    242
60120000061   60120000015    211
6738400218    60120000015    207
923330053007  60120000015    206
                            ... 
8613090369    60120000015      1
8613090866    60120000015      1
8613200560    60120000015      1
919892125680  60120000015      1
966505022318  60120000015      1
Length: 327, dtype: int64

In [12]:
print("dcs distribution split by message_type (matched rows only):")
pd.crosstab(matches["message_type"], matches["dcs"])

dcs distribution split by message_type (matched rows only):


dcs,0
message_type,
3,7153


## Full match list

In [13]:
full_match_list = matches[["time_stamp", "message_type", "dcs", "calling_gt", "called_gt", "decoded_content", "content", "reference"]]
full_match_list

,time_stamp,message_type,dcs,calling_gt,called_gt,decoded_content,content,reference
0,2026-08-02 00:01:58,3,0,60120000664,60120000015,REG-REQ?v=5;t=7DAC812049B26BC2E07C478CA1ABBE63...,d2e2b1252d467ff65e6d47efdd88c1212e2683d1724299...,3073333485
1,2026-08-02 00:02:03,3,0,60120000060,60120000015,REG-REQ?v=5;t=92F64820509544E46939605E46624FA6...,d2e2b1252d467ff65e6d47efe564461b0d2783d560b91a...,2521805300
2,2026-08-02 00:03:22,3,0,6281107018,60120000015,REG-REQ?v=5;t=522048D1725EF5D12912683B4DBB7C25...,d2e2b1252d467ff65e6d47efd56432180d478cdd64b5a2...,2514202351
3,2026-08-02 00:03:50,3,0,855970200106,60120000015,REG-REQ?v=5;t=6033D6BB4931B97A9A4307372605EC7D...,d2e2b1252d467ff65e6d47efd960b319d12614d272b398...,3083060366
4,2026-08-02 00:03:57,3,0,855970200106,60120000015,REG-REQ?v=5;t=6033D6BB4931B97A9A4307372605EC7D...,d2e2b1252d467ff65e6d47efd960b319d12614d272b398...,3089539573
...,...,...,...,...,...,...,...,...
7148,2026-08-03 23:00:48,3,0,6281107060,60120000015,REG-REQ?v=5;t=CADBBCB5E650B5B9C14DEE9490500DB1...,d2e2b1252d467ff65e6d47ef0d8344a17028ac156d3598...,2521706572
7149,2026-08-03 23:00:50,3,0,6738400118,60120000015,REG-REQ?v=3;t=199FC2BA07BE0A5F4E3959D4B7226357...,d2e2b1252d467ff6de6c47efc57239e350260cc26ec222...,3095410215
7150,2026-08-03 23:00:55,3,0,6738400118,60120000015,REG-REQ?v=3;t=199FC2BA07BE0A5F4E3959D4B7226357...,d2e2b1252d467ff6de6c47efc57239e350260cc26ec222...,3095409919
7151,2026-08-03 23:01:00,3,0,6738400118,60120000015,REG-REQ?v=3;t=199FC2BA07BE0A5F4E3959D4B7226357...,d2e2b1252d467ff6de6c47efc57239e350260cc26ec222...,2527490702


In [14]:
out_path = PROJECT_ROOT / "notebooks" / "ss7_regex_reg_req_matches.csv"
full_match_list.to_csv(out_path, index=False)
print(f"wrote {len(full_match_list)} rows to {out_path}")

wrote 7153 rows to c:\Users\IshitaGodani\Documents\projects\spam-detection-prototype\notebooks\ss7_regex_reg_req_matches.csv
